# Dataset Validation

Quick validation notebook to inspect dataset after building.

## Setup

In [ ]:
import numpy as np
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
from collections import Counter

# Import dataset loading function
import sys
sys.path.append('../')
from pipeline.io import load_processed_h5_dataset

## Load Dataset

In [ ]:
# Use the imported function
load_h5_dataset = load_processed_h5_dataset

# Update this path to your dataset
dataset_path = Path("../data/lte_wifi_dvbt/processed/lte_wifi_dvbt.h5")
# dataset_path = Path("../data/aggregation/processed/aggregated.h5")
x, y, label_names, metadata = load_h5_dataset(dataset_path)

## Dataset Overview

In [ ]:
print(f"Dataset shape: {x.shape}")
print(f"Labels shape: {y.shape}")
print(f"Number of classes: {len(label_names)}")
print(f"Classes: {label_names}")
from pathlib import Path

# Show dataset size on disk
dataset_size_bytes = Path(dataset_path).stat().st_size
# Format as human-readable
def human_readable_size(size):
    for unit in ['B','KB','MB','GB','TB']:
        if size < 1024.0:
            return f"{size:3.1f} {unit}"
        size /= 1024.0
    return f"{size:.1f} PB"

print(f"Dataset file size on disk: {human_readable_size(dataset_size_bytes)}")


## Class Distribution

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
class_counts = np.bincount(y, minlength=len(label_names))

total_windows = np.sum(class_counts, axis=0)
print(f"Total windows: {total_windows}")

plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(label_names)), class_counts, tick_label=label_names)
plt.xlabel("Class")
plt.ylabel("Number of windows")
plt.title("Class Distribution (Number of windows per class)")

# Annotate bars with counts
for rect, count in zip(bars, class_counts):
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width() / 2, height, str(count), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Metadata

In [ ]:
print("Available metadata keys:", list(metadata.keys()))
if metadata:
    for key in list(sorted(metadata.keys())):
        if key == "_global_attrs":
            print(f"\n{key}:")
            global_metadata = metadata[key]
            for key, value in global_metadata.items():
                print(f"\t{key}: {value}")
        else:
            print(f"\n{key}: shape {metadata[key].shape}, dtype {metadata[key].dtype}")

In [ ]:
# Plot the distributions for specific metadata keys

metadata_keys = ["attenuation_db", "center_frequency_hz", "gain_db", "mcs", "sample_rate_hz"]

for key in metadata_keys:
    if key not in metadata:
        print(f"Metadata key '{key}' not found.")
        continue
    
    values = metadata[key].squeeze()
    unique_vals = np.unique(values)
    plt.figure(figsize=(7, 4))
    
    if np.issubdtype(values.dtype, np.number) and len(unique_vals) > 10 and key != "mcs":
        # Continuous or many-valued numeric: plot histogram
        plt.hist(values, bins=30, color='skyblue', edgecolor='k')
        plt.xlabel(key)
        plt.ylabel("Count")
        plt.title(f"Distribution of {key}")
    else:
        # Categorical or few-valued: plot bar
        if values.ndim > 1:
            # flatten
            values = values.flatten()
        unique, counts = np.unique(values, return_counts=True)
        plt.bar(unique.astype(str), counts, color='skyblue', edgecolor='k')
        plt.xlabel(key)
        plt.ylabel("Count")
        plt.title(f"Distribution of {key}")

    plt.tight_layout()
    plt.show()


## Sample IQ Windows

In [ ]:

# Show one window from each class
n_samples_per_class = 1
fig, axes = plt.subplots(len(label_names), 1, figsize=(12, 3*len(label_names)))
if len(label_names) == 1:
    axes = [axes]

for class_id, label_name in enumerate(label_names):
    # Find first sample of this class
    class_indices = np.where(y == class_id)[0]
    if len(class_indices) == 0:
        continue
    
    idx = class_indices[np.random.randint(0, len(class_indices) - 1)]
    window = x[idx]  # (W, 2)
    
    ax = axes[class_id]
    ax.plot(window[:, 0], label="I", alpha=0.7)
    ax.plot(window[:, 1], label="Q", alpha=0.7)
    ax.set_title(f"{label_name} (class {class_id}, sample {idx})")
    ax.set_xlabel("Sample")
    ax.set_ylabel("Amplitude")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Power Distribution

In [ ]:
# Compute window power in dB for each window
# Power per window: mean(I^2 + Q^2) over window axis
window_power = np.mean(x ** 2, axis=1).sum(axis=1)  # shape: (num_windows,)
window_power_db = 10 * np.log10(window_power + 1e-12)  # add small value to avoid log(0)

# Plot histogram per class
plt.figure(figsize=(10, 6))
bins = np.linspace(np.min(window_power_db), np.max(window_power_db), 50)

for class_id, label_name in enumerate(label_names):
    class_power_db = window_power_db[y == class_id]
    plt.hist(class_power_db, bins=bins, alpha=0.5, label=label_name, density=True)

plt.xlabel("Window Power (dB)")
plt.ylabel("Density")
plt.title("Distribution of Window Power per Class")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Plot separate histogram for window power (dB) per label/class

num_classes = len(label_names)
ncols = 2
nrows = int(np.ceil(num_classes / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows), sharex=True)
axes = axes.flatten()

for class_id, label_name in enumerate(label_names):
    ax = axes[class_id]
    class_power_db = window_power_db[y == class_id]
    ax.hist(class_power_db, bins=50, alpha=0.7, color="tab:blue", density=True)
    ax.set_title(f"{label_name} (class {class_id}) - Window Power Distribution")
    ax.set_ylabel("Density")
    ax.grid(True, alpha=0.3)

# Hide any unused subplots
for unused_ax in axes[num_classes:]:
    unused_ax.axis('off')

axes[-1 if num_classes > 0 else 0].set_xlabel("Window Power (dB)")
plt.tight_layout()
plt.show()


## Spectrogram

Spectrograms of a few windows, one per class where possible.

In [ ]:
from random import choice

num_windows_to_plot = min(5, len(label_names))  # One per class, up to 5
fig, axes = plt.subplots(1, num_windows_to_plot, figsize=(5 * num_windows_to_plot, 5))

if num_windows_to_plot == 1:
    axes = [axes]

for idx, class_id in enumerate(range(num_windows_to_plot)):
    ax = axes[idx]
    # Find all indices of this class
    windows_of_class = np.where(y == class_id)[0]
    if len(windows_of_class) == 0:
        continue
    # Pick a random sample from this class
    sample_idx = choice(windows_of_class)
    window_data = x[sample_idx]

    # If complex valued: shape (N,), else: try reshape if IQ is (2, N) or (N, 2)
    if np.iscomplexobj(window_data):
        signal = window_data
    elif window_data.ndim == 2 and window_data.shape[0] == 2:
        # Assume [I; Q], shape (2, N)
        signal = window_data[0] + 1j * window_data[1]
    elif window_data.ndim == 2 and window_data.shape[1] == 2:
        # Assume shape (N, 2)
        signal = window_data[:, 0] + 1j * window_data[:, 1]
    else:
        # Fallback to treating as real signal
        signal = window_data

    # Sample rate and center frequency are per-window fields that not every
    # dataset stores; the simulated sources have neither.
    fs = float(metadata["sample_rate_hz"][sample_idx]) if "sample_rate_hz" in metadata else float("nan")
    fs = fs if np.isfinite(fs) else 1.0
    title = f"Class: {label_names[class_id]}\nIndex: {sample_idx}"
    if "center_frequency_hz" in metadata:
        center_mhz = float(metadata["center_frequency_hz"][sample_idx]) / 1e6
        title += f", C_freq: {center_mhz:g} MHz"

    ax.specgram(signal, NFFT=256, Fs=fs, noverlap=128, cmap="viridis")
    ax.set_title(title)
    ax.set_ylabel("Frequency (Hz)")
    ax.set_xlabel("Time (s)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Sanity Check: Spectrograms of Sample Windows (one per class)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()




## Normalization (if enabled)

In [ ]:
# Mean power of each window: mean(I^2 + Q^2) over its samples. A per-dataset file
# built with unit_power normalization has 1.0 for every window. In the aggregated
# file, zero-padded windows are lower by design (original_window_size / 500).
rng = np.random.default_rng(0)
sample = x[rng.choice(len(x), size=min(1000, len(x)), replace=False)].astype(np.float64)  # (n, W, 2)

power_per_window = np.mean(np.sum(sample ** 2, axis=-1), axis=-1)
print(f"Mean window power: {power_per_window.mean():.4f} "
      f"(min {power_per_window.min():.4f}, max {power_per_window.max():.4f})")

## Visual Summary

One figure per class (time domain and spectrogram), and a table of windows per
class, per location and per sample rate.

In [ ]:
# Dataset Health Report: Visual Summary
# ======================================

from random import choice
import pandas as pd

# Sample rate for the spectrograms: the most common per-window value. Simulated
# sources have none (NaN); their spectrograms use a normalized rate of 1.
sample_rate_hz = 1.0
if "sample_rate_hz" in metadata:
    sr_array = metadata["sample_rate_hz"]
    valid_sr = sr_array[~np.isnan(sr_array)]
    if len(valid_sr) > 0:
        unique_sr, counts = np.unique(valid_sr, return_counts=True)
        sample_rate_hz = float(unique_sr[np.argmax(counts)])

print(f"Sample rate used for spectrograms: {sample_rate_hz:g} Hz\n")

# ============================================================================
# 1. One figure per class: Time-domain + Spectrogram
# ============================================================================

for class_id, label_name in enumerate(label_names):
    # Find all indices of this class
    class_indices = np.where(y == class_id)[0]
    if len(class_indices) == 0:
        print(f"Warning: No samples found for class {label_name}, skipping...")
        continue
    
    # Pick a random sample from this class
    sample_idx = choice(class_indices)
    window_data = x[sample_idx]  # (W, 2)
    
    # Convert to complex signal for spectrogram
    signal = window_data[:, 0] + 1j * window_data[:, 1]
    
    # Create figure with two subplots
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Time-domain plot
    ax_time = axes[0]
    time_samples = np.arange(len(window_data))
    ax_time.plot(time_samples, window_data[:, 0], label="I", alpha=0.7, linewidth=0.8)
    ax_time.plot(time_samples, window_data[:, 1], label="Q", alpha=0.7, linewidth=0.8)
    ax_time.set_xlabel("Sample Index")
    ax_time.set_ylabel("Amplitude")
    ax_time.set_title(f"{label_name} (class {class_id}) - Time Domain\nSample index: {sample_idx}")
    ax_time.legend()
    ax_time.grid(True, alpha=0.3)
    
    # Spectrogram
    ax_spec = axes[1]
    ax_spec.specgram(signal, NFFT=256, Fs=sample_rate_hz, noverlap=128, cmap="viridis")
    ax_spec.set_xlabel("Time (s)")
    ax_spec.set_ylabel("Frequency (Hz)")
    ax_spec.set_title(f"{label_name} (class {class_id}) - Spectrogram")
    ax_spec.grid(True, alpha=0.3)
    
    # Add metadata annotation if available
    metadata_str = []
    if "center_frequency_hz" in metadata:
        cf = metadata["center_frequency_hz"][sample_idx]
        if not np.isnan(cf):
            metadata_str.append(f"Center freq: {cf/1e6:.3f} MHz")
    if "sample_rate_hz" in metadata:
        sr = metadata["sample_rate_hz"][sample_idx]
        if not np.isnan(sr):
            metadata_str.append(f"Sample rate: {sr/1e6:.3f} MHz")
    
    if metadata_str:
        ax_spec.text(0.02, 0.98, "\n".join(metadata_str), 
                    transform=ax_spec.transAxes, 
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

# ============================================================================
# 2. Summary Table: Windows per class, location, and sample rates
# ============================================================================

print("\n" + "=" * 80)
print("Dataset Health Report: Summary Tables")
print("=" * 80)

# Table 1: Windows per class
print("\n1. Windows per Class:")
print("-" * 50)
class_counts = np.bincount(y, minlength=len(label_names))
class_table_data = []
for class_id, label_name in enumerate(label_names):
    count = class_counts[class_id]
    percentage = 100 * count / len(y) if len(y) > 0 else 0
    class_table_data.append({
        "Class": label_name,
        "Class ID": class_id,
        "Windows": f"{count:,}",
        "Percentage": f"{percentage:.2f}%"
    })
df_classes = pd.DataFrame(class_table_data)
print(df_classes.to_string(index=False))

# Table 2: Windows per location (if location metadata exists)
if "location" in metadata:
    print("\n\n2. Windows per Location:")
    print("-" * 50)
    locations = metadata["location"]
    # Decode bytes to strings if needed
    if locations.dtype == object or (hasattr(locations, 'dtype') and 'U' not in str(locations.dtype)):
        locations_str = [loc.decode('utf-8') if isinstance(loc, bytes) else str(loc) 
                        for loc in locations]
    else:
        locations_str = [str(loc) for loc in locations]
    
    # Filter out empty strings
    valid_locations = [loc for loc in locations_str if loc and loc.strip()]
    
    if valid_locations:
        location_counts = Counter(valid_locations)
        location_table_data = []
        for loc, count in sorted(location_counts.items(), key=lambda x: -x[1]):
            percentage = 100 * count / len(valid_locations) if len(valid_locations) > 0 else 0
            location_table_data.append({
                "Location": loc,
                "Windows": f"{count:,}",
                "Percentage": f"{percentage:.2f}%"
            })
        df_locations = pd.DataFrame(location_table_data)
        print(df_locations.to_string(index=False))
    else:
        print("No valid location data found.")
else:
    print("\n\n2. Windows per Location:")
    print("-" * 50)
    print("Location metadata not available in this dataset.")

# Table 3: Sample rate distribution
print("\n\n3. Sample Rate Distribution:")
print("-" * 50)
if "sample_rate_hz" in metadata:
    sr_array = metadata["sample_rate_hz"]
    valid_sr = sr_array[~np.isnan(sr_array)]
    if len(valid_sr) > 0:
        sr_counts = Counter(valid_sr)
        sr_table_data = []
        total_valid = len(valid_sr)
        for sr_hz, count in sorted(sr_counts.items()):
            sr_mhz = sr_hz / 1e6
            percentage = 100 * count / total_valid if total_valid > 0 else 0
            sr_table_data.append({
                "Sample Rate (MHz)": f"{sr_mhz:.3f}",
                "Windows": f"{count:,}",
                "Percentage": f"{percentage:.2f}%"
            })
        df_sr = pd.DataFrame(sr_table_data)
        print(df_sr.to_string(index=False))
    else:
        print("All sample rate values are NaN.")
else:
    # Try to get from global attributes
    with h5py.File(dataset_path, "r") as f:
        if "sample_rate_hz" in f.attrs:
            sr = float(f.attrs["sample_rate_hz"])
            sr_mhz = sr / 1e6
            total_windows = len(y)
            print(f"Fixed sample rate: {sr_mhz:.3f} MHz ({total_windows:,} windows)")
        else:
            print("Sample rate information not available.")

# Overall summary
print("\n\n" + "=" * 80)
print("Overall Summary:")
print("-" * 80)
print(f"Total windows: {len(y):,}")
print(f"Number of classes: {len(label_names)}")
print(f"Window size: {x.shape[1]} samples")
if "sample_rate_hz" in metadata:
    sr_array = metadata["sample_rate_hz"]
    valid_sr = sr_array[~np.isnan(sr_array)]
    if len(valid_sr) > 0:
        unique_sr = np.unique(valid_sr)
        if len(unique_sr) == 1:
            print(f"Sample rate: {unique_sr[0]/1e6:.3f} MHz (fixed)")
        else:
            print(f"Sample rates: {len(unique_sr)} unique values")
            print(f"  Range: {np.min(unique_sr)/1e6:.3f} - {np.max(unique_sr)/1e6:.3f} MHz")
else:
    with h5py.File(dataset_path, "r") as f:
        if "sample_rate_hz" in f.attrs:
            sr = float(f.attrs["sample_rate_hz"])
            print(f"Sample rate: {sr/1e6:.3f} MHz (from global metadata)")

print("=" * 80)